# 05 — Log-Linear Models

Two separate log-linear analyses:

**Analysis A — Feature × label tables**
Three-way Poisson GLM on e.g. `L_R × category × state`. Tests conditional
independence and homogeneous association in the feature space.

**Analysis B — Label joint distribution**
Log-linear model on the 2^5 = 32-cell table of label combinations.
Tests pairwise and higher-order conditional independence between fraud types.
Output: label association graph.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2
import sys
sys.path.insert(0, '..')

from src.labels import LABEL_COLS

In [ ]:
df = pd.read_parquet('../data/processed/train_labeled.parquet')

## Analysis A: L_R × category × state

In [ ]:
# Reduce state to top-10 by volume to keep table manageable
top_states = df['state'].value_counts().head(10).index
df_a = df[df['state'].isin(top_states)].copy()
df_a['L_R_cat'] = df_a['L_R'].astype(str)

cells_a = (
    df_a.groupby(['L_R_cat', 'category', 'state'])
    .size()
    .reset_index(name='n')
)
print(f'Cell table shape: {cells_a.shape}')
cells_a.head()

In [ ]:
# Saturated model (all two-way interactions, no 3-way)
m_pairwise = smf.glm(
    'n ~ L_R_cat*category + L_R_cat*state + category*state',
    data=cells_a, family=sm.families.Poisson()
).fit()

# Saturated (including 3-way)
m_saturated = smf.glm(
    'n ~ L_R_cat*category*state',
    data=cells_a, family=sm.families.Poisson()
).fit()

# LR test: does 3-way term matter?
g2 = 2 * (m_saturated.llf - m_pairwise.llf)
df_diff = m_saturated.df_model - m_pairwise.df_model
p = chi2.sf(g2, df_diff)
print(f'3-way interaction test: G²={g2:.2f}  df={df_diff}  p={p:.4f}')
print('Conclusion: 3-way interaction', 'IS' if p<0.05 else 'IS NOT', 'significant')

## Analysis B: Label joint distribution (2^5 = 32 cells)

In [ ]:
label_counts = (
    df.groupby(LABEL_COLS)
    .size()
    .reset_index(name='n')
)
# Cast labels to strings for formula interface
for col in LABEL_COLS:
    label_counts[col] = label_counts[col].astype(str)

print(f'Non-zero label combinations: {len(label_counts)} / 32')
label_counts.sort_values('n', ascending=False).head(10)

In [ ]:
# Mutual independence model
m_indep = smf.glm(
    'n ~ L_V + L_G + L_C + L_R + L_T',
    data=label_counts, family=sm.families.Poisson()
).fit()

# All pairwise interactions
m_pairwise_labels = smf.glm(
    'n ~ (L_V + L_G + L_C + L_R + L_T)**2',
    data=label_counts, family=sm.families.Poisson()
).fit()

# Saturated
m_sat_labels = smf.glm(
    'n ~ L_V*L_G*L_C*L_R*L_T',
    data=label_counts, family=sm.families.Poisson()
).fit()

for name, m in [('Independence', m_indep), ('Pairwise', m_pairwise_labels), ('Saturated', m_sat_labels)]:
    g2_dev = m.deviance
    df_dev = m.df_resid
    p_dev  = chi2.sf(g2_dev, df_dev)
    print(f'{name:12s}: deviance={g2_dev:.2f}  df={df_dev}  p={p_dev:.4f}')

In [ ]:
# Pairwise interaction coefficients → association graph weights
coefs = m_pairwise_labels.params
interactions = {k: v for k, v in coefs.items() if ':' in k}
print('Pairwise interaction log-rates:')
for k, v in sorted(interactions.items(), key=lambda x: abs(x[1]), reverse=True):
    print(f'  {k}: {v:.4f}  (OR={np.exp(v):.4f})')